In [1]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

In [2]:
X_train = joblib.load("../data/processed/X_train.pkl")
X_test = joblib.load("../data/processed/X_test.pkl")

y_train = joblib.load("../data/processed/y_train.pkl")
y_test = joblib.load("../data/processed/y_test.pkl")

print("Processed datasets loaded successfully!\n")

print(f"Training Features : {X_train.shape}")
print(f"Testing Features  : {X_test.shape}")
print(f"Training Labels   : {y_train.shape}")
print(f"Testing Labels    : {y_test.shape}")

Processed datasets loaded successfully!

Training Features : (2400, 137)
Testing Features  : (600, 137)
Training Labels   : (2400,)
Testing Labels    : (600,)


In [3]:
best_model = joblib.load("../models/best_model.pkl")

print("Current Best Model Loaded Successfully!")
print(best_model)

Current Best Model Loaded Successfully!
DecisionTreeClassifier(max_depth=10, random_state=42)


In [4]:
# Decision Tree Hyperparameter Tuning

dt = DecisionTreeClassifier(random_state=RANDOM_STATE)

param_grid_dt = {
    "criterion": ["gini", "entropy"],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [None, "sqrt", "log2"]
}

grid_dt = GridSearchCV(
    estimator=dt,
    param_grid=param_grid_dt,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_

print("Best Decision Tree Parameters")
print(grid_dt.best_params_)
print("\nBest Cross Validation Accuracy:")
print(f"{grid_dt.best_score_:.4f}")

Best Decision Tree Parameters
{'criterion': 'gini', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2}

Best Cross Validation Accuracy:
0.3517


In [5]:
# Evaluate Tuned Decision Tree

y_pred_dt = best_dt.predict(X_test)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
dt_precision = precision_score(y_test, y_pred_dt, average="weighted")
dt_recall = recall_score(y_test, y_pred_dt, average="weighted")
dt_f1 = f1_score(y_test, y_pred_dt, average="weighted")

print("Tuned Decision Tree Results")

print(f"Accuracy : {dt_accuracy:.4f}")
print(f"Precision: {dt_precision:.4f}")
print(f"Recall   : {dt_recall:.4f}")
print(f"F1 Score : {dt_f1:.4f}")

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_dt))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred_dt))

Tuned Decision Tree Results
Accuracy : 0.3417
Precision: 0.3339
Recall   : 0.3417
F1 Score : 0.3321

Classification Report

              precision    recall  f1-score   support

           0       0.37      0.50      0.43       207
           1       0.32      0.31      0.31       196
           2       0.30      0.21      0.25       197

    accuracy                           0.34       600
   macro avg       0.33      0.34      0.33       600
weighted avg       0.33      0.34      0.33       600


Confusion Matrix

[[103  61  43]
 [ 82  60  54]
 [ 91  64  42]]


In [15]:
# Random Forest Hyperparameter Tuning

rf = RandomForestClassifier(random_state=RANDOM_STATE)

param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

print("Best Random Forest Parameters")
print(grid_rf.best_params_)

print("\nBest Cross Validation Accuracy:")
print(f"{grid_rf.best_score_:.4f}")

Best Random Forest Parameters
{'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 200}

Best Cross Validation Accuracy:
0.3471


In [16]:
# Evaluate Tuned Random Forest

y_pred_rf = best_rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf, average="weighted")
rf_recall = recall_score(y_test, y_pred_rf, average="weighted")
rf_f1 = f1_score(y_test, y_pred_rf, average="weighted")

print("Tuned Random Forest Results")

print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1 Score : {rf_f1:.4f}")

print("\nClassification Report\n")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, y_pred_rf))

Tuned Random Forest Results
Accuracy : 0.3300
Precision: 0.3287
Recall   : 0.3300
F1 Score : 0.3214

Classification Report

              precision    recall  f1-score   support

           0       0.34      0.48      0.40       207
           1       0.34      0.26      0.29       196
           2       0.31      0.24      0.27       197

    accuracy                           0.33       600
   macro avg       0.33      0.33      0.32       600
weighted avg       0.33      0.33      0.32       600


Confusion Matrix

[[100  55  52]
 [ 90  50  56]
 [108  41  48]]


In [17]:
comparison_improved = pd.DataFrame({
    "Model": [
        "Original Decision Tree",
        "Tuned Decision Tree",
        "Original Random Forest",
        "Tuned Random Forest"
    ],
    "Accuracy": [
        0.368333,
        dt_accuracy,
        0.351667,
        rf_accuracy
    ],
    "Precision": [
        0.369713,
        dt_precision,
        0.349016,
        rf_precision
    ],
    "Recall": [
        0.368333,
        dt_recall,
        0.351667,
        rf_recall
    ],
    "F1 Score": [
        0.354209,
        dt_f1,
        0.346123,
        rf_f1
    ]
})

comparison_improved = comparison_improved.sort_values(
    by="Accuracy",
    ascending=False
)

display(comparison_improved)

,Model,Accuracy,Precision,Recall,F1 Score
0,Original Decision Tree,0.368333,0.369713,0.368333,0.354209
2,Original Random Forest,0.351667,0.349016,0.351667,0.346123
1,Tuned Decision Tree,0.341667,0.333905,0.341667,0.332113
3,Tuned Random Forest,0.330000,0.328670,0.330000,0.321442


In [18]:
if rf_accuracy > dt_accuracy:
    final_model = best_rf
    final_model_name = "Random Forest"
    final_accuracy = rf_accuracy
else:
    final_model = best_dt
    final_model_name = "Decision Tree"
    final_accuracy = dt_accuracy

print("FINAL MODEL")
print(final_model_name)
print(f"Accuracy : {final_accuracy:.4f}")

FINAL MODEL
Decision Tree
Accuracy : 0.3417


In [24]:
# Compare Original Decision Tree vs Tuned Random Forest

print("Decision Tree Accuracy :", dt_accuracy)
print("Random Forest Accuracy :", rf_accuracy)
if rf_accuracy > dt_accuracy:
    final_model = best_rf
    final_model_name = "Random Forest"
else:
    final_model = joblib.load("../models/best_model.pkl")
    final_model_name = "Original Decision Tree"

print(f"\nFinal Production Model : {final_model_name}")

Decision Tree Accuracy : 0.3416666666666667
Random Forest Accuracy : 0.33

Final Production Model : Original Decision Tree


In [20]:
scores = cross_val_score(
    final_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Scores")
print(scores)

print(f"\nMean Accuracy : {scores.mean():.4f}")
print(f"Std Deviation : {scores.std():.4f}")

Cross Validation Scores
[0.275      0.29375    0.35416667 0.34583333 0.35416667]

Mean Accuracy : 0.3246
Std Deviation : 0.0335


In [21]:
joblib.dump(
    final_model,
    "../models/final_model.pkl"
)

print("Final production model saved successfully!")

Final production model saved successfully!


In [22]:
comparison_improved.to_csv(
    "../models/model_improvement_results.csv",
    index=False
)

print("Model improvement results saved successfully!")

Model improvement results saved successfully!


In [23]:
print("MODEL IMPROVEMENT COMPLETED")

print(f"Final Model : {final_model_name}")

print("\nFiles Generated:")
print("final_model.pkl")
print("model_improvement_results.csv")

MODEL IMPROVEMENT COMPLETED
Final Model : Original Decision Tree

Files Generated:
final_model.pkl
model_improvement_results.csv
